In [1]:
4+4

8

In [2]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver, InMemorySaver

from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def book_meeting(person: str, day: str) -> str:
    """Book a meeting with a person on a given day"""
    return f'meeting booked with {person} on {day}'

approval_agent = create_agent(
    model='openai:gpt-3.5-turbo',
    tools=[book_meeting],
    system_prompt="You are a scheduling assistant, use the book_meeting tool.",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"book_meeting":True})],
    checkpointer=InMemorySaver()
)

config = {"configurable":{'thread_id':"approval-demo"}}
result = await approval_agent.ainvoke(
    {"messages":[{"role":"user","content":"Book a meeting with Sam on Friday"}]}, config
)

interrupt = result["__interrupt__"][0]
print("The agent paused and is asking for approval")
print(interrupt.value['action_requests'][0]["description"])

The agent paused and is asking for approval
Tool execution requires approval

Tool: book_meeting
Args: {'person': 'Sam', 'day': 'Friday'}


In [3]:
config={'configurable':{'thread_id':"rejection-demo"}}
result=await approval_agent.ainvoke(
    {'messages':[{"role":"user","content":"Book a meeting with Sam on Friday"}]},config
)

interrupt=result['__interrupt__'][0]
print("The agent paused and is asking for approval:")
print(interrupt.value['action_requests'][0]['description'])

resumed = await approval_agent.ainvoke(Command(resume={'decisions':[{"type":"reject","message":"Rejected -no meetings on Friday"}]}), config)
print(resumed['messages'][-1].content)

The agent paused and is asking for approval:
Tool execution requires approval

Tool: book_meeting
Args: {'person': 'Sam', 'day': 'Friday'}
I'm sorry, but I couldn't schedule a meeting with Sam on Friday. Is there another day that works for you?
